# Data Quality Monitoring System for E-Commerce Operations

## Project Overview

This project presents the development of a Data Quality Monitoring System using the **Olist Brazilian E-Commerce Public Dataset**. The objective is to simulate a real-world business analytics workflow by identifying, assessing, and improving the quality of transactional data before it is used for reporting and decision-making.

The project follows an end-to-end data analytics pipeline, beginning with data preparation in **Python**, followed by data modelling and quality analysis in **PostgreSQL**, and concluding with interactive **Power BI** dashboards that monitor key data quality metrics and business performance indicators.

Throughout the project, data quality dimensions such as **completeness, accuracy, consistency, validity, uniqueness, and timeliness** are evaluated. The datasets are cleaned, standardized, validated, and transformed into analysis-ready data to support reliable business intelligence and operational reporting.

This project demonstrates practical skills in data cleaning, exploratory data analysis, SQL development, data quality assessment, and dashboard development while following industry-standard data analytics practices.

This script focuses on the preparation of the Olist order reviews dataset as part of the Data Quality Monitoring System for E-Commerce Operations. The cleaned dataset supports customer satisfaction and service quality analysis by capturing review scores, customer feedback, review creation dates, and response timestamps associated with completed orders. Preparing this dataset ensures that customer review information is complete, consistent, and reliable for downstream analysis in PostgreSQL, SQL, and Power BI, enabling accurate evaluation of customer sentiment, review trends, and overall service performance.

### Import The Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re 
import psycopg2
from sqlalchemy import create_engine 
from pathlib import Path

### Load The Dataset

In [3]:
order_reviews_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\raw\olist_order_reviews_dataset.csv")

## **ORDER DATASET**

### 1. Data Inspection

In [ ]:
# The first five rows of the order reviews dataset
order_reviews_df.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


In [ ]:
# The number of rows and columns in the order reviews dataset
order_reviews_df.shape

(99224, 7)

In [6]:
#The column names and their data types in the order reviews dataset
order_reviews_df.dtypes

review_id                  object
order_id                   object
review_score                int64
review_comment_title       object
review_comment_message     object
review_creation_date       object
review_answer_timestamp    object
dtype: object

The order reviews dataset contains seven variables describing customer review information associated with completed orders. The `review_id`, `order_id`, `review_comment_title`, and `review_comment_message` columns are stored as text (`object`), while `review_score` is stored as an integer (`int64`). Both `review_creation_date` and `review_answer_timestamp` are currently stored as text (`object`) and will require conversion to the appropriate datetime data type during data cleaning.

In [8]:
# The number of missing values in each column of the order reviews dataset
order_reviews_df.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

The inspection identified no missing values in the primary identifier fields, review score, or review date variables, indicating that all review records contain the essential information required for analysis. Missing values are present only in the `review_comment_title` and `review_comment_message` columns, with 87,656 and 58,247 missing values respectively.

These fields capture optional written feedback provided by customers. Since customers can submit a review score without providing a title or written comment, the observed missing values represent expected business behaviour rather than data quality issues.

In [9]:
# The number of duplicate rows in the order payments dataset columns
for col in order_reviews_df.columns:
    duplicates = order_reviews_df[col].duplicated().sum()
    print(f"{col}: {duplicates}")

review_id: 814
order_id: 551
review_score: 99219
review_comment_title: 94696
review_comment_message: 63064
review_creation_date: 98588
review_answer_timestamp: 976


The inspection identified duplicate values across several columns, including the review identifiers, order identifiers, review scores, review comment titles, review comment messages, review creation dates, and review response timestamps. These repeated values are expected within transactional datasets because multiple reviews may share the same rating, comment, submission date, or response timestamp.

Duplicate values within individual columns do not necessarily indicate duplicated records. A complete duplicate record assessment will therefore be conducted during the data profiling phase to determine whether identical rows exist within the dataset.

In [ ]:
# Statistical summary of the order reviews dataset numeric column
order_reviews_df.describe()

,review_score
count,99224.000000
mean,4.086421
std,1.347579
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


Reasoning

The numeric summary indicates that the dataset contains **99,224** review records with review scores ranging from **1** to **5**, confirming that all ratings fall within the expected Olist review scale. The average review score is approximately **4.09**, while both the median and upper quartile equal **5**, indicating that most customers assigned high ratings. The standard deviation of approximately **1.35** suggests moderate variation in customer ratings. No review scores fall outside the expected range, and no numerical anomalies requiring further investigation were identified during the inspection stage.

In [12]:
# Statistical summary of the order reviews dataset categorical columns
order_reviews_df.describe(include='object')

,review_id,order_id,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
count,99224,99224,11568,40977,99224,99224
unique,98410,98673,4527,36159,636,98248
top,7b606b0d57b078384f0b58eac1d41d78,c88b1d1b157a9999ce368f218a407141,Recomendo,Muito bom,2017-12-19 00:00:00,2017-06-15 23:21:05
freq,3,3,423,230,463,4


The categorical summary shows that both the `review_id` and `order_id` columns contain a high number of unique values, which is consistent with transactional review data. The number of unique values is slightly lower than the total record count, indicating that some identifiers appear more than once and should be investigated during the data profiling phase.

The `review_comment_title` and `review_comment_message` columns contain substantially fewer unique values due to the large number of missing values and repeated customer responses. This behaviour is expected because providing written feedback is optional and customers may submit identical comments.

Similarly, the `review_creation_date` and `review_answer_timestamp` columns contain repeated values, reflecting multiple reviews being created or answered on the same dates and times. No unexpected categories or inconsistencies were identified during this inspection stage.

### 2. Data Profiling

#### Duplicated Records

In [13]:
# Count duplicate records
duplicate_count = order_reviews_df.duplicated().sum()

print(f"Duplicate Records: {duplicate_count}")

Duplicate Records: 0


In [15]:
# Display the dimensions of the duplicate records
duplicate_records.shape

(0, 7)

In [16]:
# Compare original and deduplicated datasets
print(f"Original Records : {len(order_reviews_df):,}")
print(f"Unique Records   : {len(order_reviews_df.drop_duplicates()):,}")
print(f"Duplicate Records: {duplicate_count:,}")

Original Records : 99,224
Unique Records   : 99,224
Duplicate Records: 0


The duplicate record assessment confirmed that the order reviews dataset contains no duplicate records. A total of **99,224** records were examined, with the duplicate count equal to **0**. Additionally, the deduplicated dataset retained all **99,224** records, confirming that no identical observations exist within the dataset. Consequently, no duplicate records require removal, and all records will be retained in the cleaned dataset.

#### Review Identifiers

In [17]:
# Count duplicated review IDs
duplicated_review_ids = (
    order_reviews_df["review_id"]
    .duplicated()
    .sum()
)

print(f"Duplicated Review IDs: {duplicated_review_ids}")

Duplicated Review IDs: 814


In [18]:
# Display duplicated review IDs
duplicated_review_records = order_reviews_df[
    order_reviews_df["review_id"].duplicated(keep=False)
].sort_values("review_id")

duplicated_review_records

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
...,...,...,...,...,...,...,...
31120,fe5c833752953fed3209646f1f63b53c,4863e15fa53273cc7219c58f5ffda4fb,1,NaN,"Comprei dois produtos e ambos, mesmo enviados ...",2018-02-28 00:00:00,2018-02-28 13:57:52
7870,ff2fc9e68f8aabfbe18d710b83aabd30,2da58e0a7dcfa4ce1e00fad9d03ca3b5,2,NaN,NaN,2018-03-17 00:00:00,2018-03-19 11:44:15
82521,ff2fc9e68f8aabfbe18d710b83aabd30,1078d496cc6ab9a8e6f2be77abf5091b,2,NaN,NaN,2018-03-17 00:00:00,2018-03-19 11:44:15
73951,ffb8cff872a625632ac983eb1f88843c,c44883fc2529b4aa03ca90e7e09d95b6,3,NaN,NaN,2017-07-22 00:00:00,2017-07-26 13:41:07


In [19]:
# Number of unique duplicated review IDs
duplicated_review_records["review_id"].nunique()

789

In [20]:
# Frequency of duplicated review IDs
duplicated_review_records["review_id"].value_counts()

review_id
4548534449b1f572e357211b90724f1b    3
2d6ac45f859465b5c185274a1c929637    3
1fb4ddc969e6bea80e38deec00393a6f    3
4d0e6dd087008d1f992d25ef6e1f619f    3
c444278834184f72b1484dfe47de7f97    3
                                   ..
5bdf704ce1edc91bc6c73abede903d1c    2
5c4bd938f98283c5d7145a9d25a89c3e    2
5c99f6ff0f883ea3b283853720266109    2
5ccf9e14796ecb4f98ce11d896e9f0c0    2
ffb8cff872a625632ac983eb1f88843c    2
Name: count, Length: 789, dtype: int64

In [21]:
# Compare duplicated review IDs with associated order IDs
duplicated_review_records[
    ["review_id", "order_id"]
].sort_values(["review_id", "order_id"])

,review_id,order_id
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f
...,...,...
40378,fe5c833752953fed3209646f1f63b53c,d3775e436e60258e62e678a0f68a0f8d
82521,ff2fc9e68f8aabfbe18d710b83aabd30,1078d496cc6ab9a8e6f2be77abf5091b
7870,ff2fc9e68f8aabfbe18d710b83aabd30,2da58e0a7dcfa4ce1e00fad9d03ca3b5
73951,ffb8cff872a625632ac983eb1f88843c,c44883fc2529b4aa03ca90e7e09d95b6


In [22]:
# Determine whether duplicated review IDs contain identical records
(
    duplicated_review_records
    .groupby("review_id")
    .nunique()
)

,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
review_id,,,,,,
00130cbe1f9d422698c812ed8ded1919,2,1,0,1,1,1
0115633a9c298b6a98bcbe4eee75345f,2,1,0,0,1,1
0174caf0ee5964646040cd94e15ac95e,2,1,0,1,1,1
017808d29fd1f942d97e50184dfb4c13,2,1,0,0,1,1
0254bd905dc677a6078990aad3331a36,2,1,0,1,1,1
...,...,...,...,...,...,...
fde2e6abaf5bb64f7407a44741c24dec,2,1,0,0,1,1
fde5986d35c89aa1b6ce4149de82a0d3,2,1,0,0,1,1
fe5c833752953fed3209646f1f63b53c,2,1,0,1,1,1


In [23]:
# Display one duplicated review ID in full for detailed inspection
sample_review_id = duplicated_review_records["review_id"].iloc[0]

order_reviews_df[
    order_reviews_df["review_id"] == sample_review_id
]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23


In [25]:
duplicated_review_records["review_id"].value_counts().value_counts()

count
2    764
3     25
Name: count, dtype: int64

The investigation identified **814 duplicated review ID** occurrences, representing **789 unique review IDs**. Further analysis revealed that duplicated review IDs are associated with different order IDs, indicating that the same review identifier appears across multiple orders rather than being repeated within the same order.

Inspection of the duplicated records shows that, for each duplicated review ID, the review score, review title, review message, review creation date, and review response timestamp remain identical. The only attribute that varies is the `order_id`. This finding is supported by the uniqueness assessment, which indicates that duplicated review IDs have two unique order IDs while all remaining attributes contain only a single unique value.

Additionally, the frequency analysis shows that most duplicated review IDs occur **twice**, while a small number occur **three times**. Since the duplicated review records are not identical rows and reference different orders, they do not constitute duplicate records. Instead, they represent a characteristic of the source data in which the same customer review has been linked to multiple orders.

The frequency analysis shows that the majority of duplicated review IDs occur **twice**, accounting for **764** review identifiers. Additionally, **25** review identifiers occur **three** times. This indicates that duplicated review IDs are relatively uncommon and are primarily limited to two associated orders, with only a small number linked to three orders.